In [1]:
!pip uninstall -y transformers tokenizers
!pip install transformers==4.46.3 tokenizers==0.20.3

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 64.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [2]:
import transformers
import tokenizers

print(transformers.__version__)
print(tokenizers.__version__)

4.46.3
0.20.3


In [3]:
!git clone https://github.com/FlagOpen/FlagEmbedding.git
%cd FlagEmbedding
!pip install -e .

Cloning into 'FlagEmbedding'...
remote: Enumerating objects: 11544, done.
remote: Counting objects: 100% (3549/3549), done.
remote: Compressing objects: 100% (1262/1262), done.
remote: Total 11544 (delta 2338), reused 2287 (delta 2287), pack-reused 7995 (from 1)
Receiving objects: 100% (11544/11544), 51.36 MiB | 28.88 MiB/s, done.
Resolving deltas: 100% (6593/6593), done.
/kaggle/working/FlagEmbedding
Obtaining file:///kaggle/working/FlagEmbedding
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 16.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.8 MB/s eta 0:00:00
  Running setup.py develop for FlagEmbedding


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("madhavkumar244/madhavscuadhn")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/madhavkumar244/madhavscuadhn


In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.3 MB/s eta 0:00:00:00:0100:01


In [6]:
import transformers
import tokenizers

print(transformers.__version__)
print(tokenizers.__version__)

4.46.3
0.20.3


In [8]:
from pathlib import Path

path = Path("/kaggle/working/FlagEmbedding/FlagEmbedding/inference/embedder/encoder_only/base.py")

text = path.read_text()

text = text.replace(
    "dtype=self.get_model_torch_dtype(),",
    "torch_dtype=self.get_model_torch_dtype(),"
)

path.write_text(text)

print("Patched base.py")

Patched base.py


In [13]:
!python /kaggle/input/datasets/madhavkumar244/madhavscuadhn/create_reranker_train.py

Loading retriever...
Retriever loaded.

100%|█████████████████████████████████████████| 408/408 [09:20<00:00,  1.37s/it]

Saved 11180 samples
Output : /kaggle/working/train_bge_same_contract_hn.jsonl


In [26]:
!python -m FlagEmbedding.finetune.reranker.encoder_only.base \
--model_name_or_path BAAI/bge-reranker-base \
--cache_dir /root/.cache/huggingface/hub \
--train_data /kaggle/working/train_bge_same_contract_hn.jsonl \
--cache_path /kaggle/working/cache \
--train_group_size 8 \
--query_max_len 128 \
--passage_max_len 384 \
--pad_to_multiple_of 8 \
--output_dir /kaggle/working/bge_reranker \
--learning_rate 1e-5 \
--fp16 \
--num_train_epochs 2 \
--per_device_train_batch_size 8 \
--dataloader_drop_last True \
--warmup_ratio 0.1 \
--gradient_checkpointing \
--logging_steps 20 \
--save_strategy steps \
--save_steps 500 \
--save_total_limit 3 \
--report_to none

/kaggle/working/FlagEmbedding/FlagEmbedding/finetune/reranker/encoder_only/base/runner.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `EncoderOnlyRerankerTrainer.__init__`. Use `processing_class` instead.
  trainer = EncoderOnlyRerankerTrainer(
  0%|                                                  | 0/2794 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2852: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
{'loss': 2.5648, 'grad_norm': 47.75328063964844, 'learning_rate': 6.071428571428572e-07, 'epoch': 0.01}
{'loss': 2.2388, 'grad_norm': 53.53207778930664,

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSequenceClassification

login("YOUR_HUGGINGFACE_TOKEN")  # or use login() and paste the token

MODEL_DIR = "/kaggle/working/bge_reranker"
REPO_ID = "Madhav2832005/bge-base-reranker-finetuned"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

tokenizer.push_to_hub(REPO_ID)
model.push_to_hub(REPO_ID)

print("Upload complete!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete!


In [40]:
!python /kaggle/input/datasets/madhavkumar244/madhavscuadhn/evaluate_reranker.py

tokenizer_config.json: 1.28kB [00:00, 6.01MB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 6.12MB/s]
tokenizer.json: 100%|██████████████████████| 17.1M/17.1M [00:00<00:00, 27.8MB/s]
config.json: 100%|█████████████████████████████| 811/811 [00:00<00:00, 5.68MB/s]
model.safetensors: 100%|███████████████████| 1.11G/1.11G [00:15<00:00, 73.1MB/s]
Loading Retriever...
tokenizer_config.json: 1.29kB [00:00, 6.93MB/s]
vocab.txt: 232kB [00:00, 13.4MB/s]
special_tokens_map.json: 100%|█████████████████| 695/695 [00:00<00:00, 6.54MB/s]
✓ Retriever loaded

Contracts : 102

Evaluating Contracts:   0%|                             | 0/102 [00:00<?, ?it/s]You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the 